# 🎞️ ROTBOTS — Assemble
## Combine Videos + Audio into Final Video

---

This is the final step! FFmpeg combines everything into one video:

1. **Concatenate** video scenes in order
2. **Overlay** narration audio on each scene
3. **Export** the final video

> **🤔 Think about:** How does the order of scenes change the meaning? What does the "cut" do?

---

## 🔧 Setup

In [ ]:
# ============================================================
# SETUP
# ============================================================

import json
import subprocess
from pathlib import Path
from IPython.display import display, Markdown, Video, HTML

# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
VIDEOS_DIR = WORK_DIR / 'videos'
AUDIO_DIR = WORK_DIR / 'audio'

# Check what we have
print('📁 Checking workspace...')
video_files = sorted(VIDEOS_DIR.glob('scene_*.mp4')) if VIDEOS_DIR.exists() else []
audio_files = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []

print(f'   🎬 Videos: {len(video_files)} clips')
for v in video_files: print(f'      {v.name} ({v.stat().st_size/1024:.0f}KB)')
print(f'   🎙️ Audio: {len(audio_files)} narrations')
for a in audio_files: print(f'      {a.name} ({a.stat().st_size/1024:.0f}KB)')

if not video_files:
    print('\n❌ No videos found! Run 05_Generate.ipynb first.')
else:
    print(f'\n✅ Ready to assemble!')

In [ ]:
# ============================================================
# HELPER: Get video/audio duration with ffprobe
# ============================================================

def get_duration(path):
    try:
        result = subprocess.run(
            ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', str(path)],
            capture_output=True, text=True, timeout=10
        )
        return float(result.stdout.strip())
    except:
        return 5.0

def run_ffmpeg(cmd, description=''):
    """Run an FFmpeg command and show progress."""
    if description:
        print(f'   {description}...', end=' ', flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if result.returncode == 0:
        print('✅' if description else '', end='')
        return True
    else:
        print(f'❌ FFmpeg error: {result.stderr[:200]}')
        return False

print('✅ FFmpeg helpers loaded')

---
## 🎞️ Station 6: Assemble

This combines all video clips into one video, optionally overlaying narration audio.

In [ ]:
# ============================================================
# STEP 1: Normalize all videos to same format
# ============================================================

print('🔧 Normalizing video formats...')
TEMP_DIR = Path('/content/temp_assembly')
TEMP_DIR.mkdir(exist_ok=True)

normalized_videos = []
for v in video_files:
    out = TEMP_DIR / v.name
    # Normalize to 720p, 24fps, yuv420p for compatibility
    cmd = [
        'ffmpeg', '-y', '-i', str(v),
        '-vf', 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black',
        '-r', '24', '-pix_fmt', 'yuv420p',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
        '-an',  # Remove audio from source videos
        str(out)
    ]
    if run_ffmpeg(cmd, f'Normalizing {v.name}'):
        normalized_videos.append(out)

print(f'\n✅ Normalized {len(normalized_videos)} videos')

In [ ]:
# ============================================================
# STEP 2: Concatenate all videos
# ============================================================

print('🎬 Concatenating videos...')

# Create concat file
concat_file = TEMP_DIR / 'concat.txt'
with open(concat_file, 'w') as f:
    for v in normalized_videos:
        f.write(f"file '{v}'\n")

concat_output = TEMP_DIR / 'concatenated.mp4'
cmd = [
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', str(concat_file),
    '-c', 'copy',
    str(concat_output)
]

if run_ffmpeg(cmd, 'Concatenating all scenes'):
    duration = get_duration(concat_output)
    print(f'   Total video duration: {duration:.1f}s')

In [ ]:
# ============================================================
# STEP 3: Mix narration audio (if available)
# ============================================================

final_output = WORK_DIR / 'final_video.mp4'

if audio_files:
    print('🎙️ Adding narration audio...')

    # First, concatenate all narration audio with gaps
    # Load audio manifest for timing
    manifest_path = WORK_DIR / 'audio_manifest.json'
    if manifest_path.exists():
        with open(manifest_path) as f:
            audio_manifest = json.load(f)
    
    # Simple approach: concatenate all narration audio files
    audio_concat_file = TEMP_DIR / 'audio_concat.txt'
    with open(audio_concat_file, 'w') as f:
        for a in audio_files:
            f.write(f"file '{a}'\n")

    combined_audio = TEMP_DIR / 'combined_narration.mp3'
    run_ffmpeg([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', str(audio_concat_file),
        '-c', 'copy',
        str(combined_audio)
    ], 'Combining narration audio')

    # Overlay narration on video
    cmd = [
        'ffmpeg', '-y',
        '-i', str(concat_output),
        '-i', str(combined_audio),
        '-c:v', 'copy',
        '-c:a', 'aac', '-b:a', '192k',
        '-map', '0:v:0', '-map', '1:a:0',
        '-shortest',
        str(final_output)
    ]
    run_ffmpeg(cmd, 'Mixing video + narration')
else:
    print('ℹ️ No narration audio found — creating video-only output')
    # Just copy the concatenated video
    import shutil
    shutil.copy2(concat_output, final_output)

if final_output.exists():
    size_mb = final_output.stat().st_size / (1024 * 1024)
    duration = get_duration(final_output)
    print(f'\n✅ Final video created!')
    print(f'   📁 {final_output}')
    print(f'   📐 Duration: {duration:.1f}s')
    print(f'   💾 Size: {size_mb:.1f}MB')

---
## 🎬 Watch Your Video!

In [ ]:
# ============================================================
# PLAY THE FINAL VIDEO
# ============================================================

if final_output.exists():
    display(Markdown('# 🎬 Your AI-Generated Video'))
    display(Markdown(f'*Created by the ROTBOTS content machine*'))
    display(Markdown('---'))
    try:
        display(Video(str(final_output), embed=True, width=720))
    except:
        print(f'Cannot preview inline. File saved at: {final_output}')
        print(f'Download it from Google Drive: My Drive/rotbots_workshop/final_video.mp4')
else:
    print('❌ No final video found')

In [ ]:
# ============================================================
# DOWNLOAD (optional)
# ============================================================

DOWNLOAD_LOCALLY = False  # Set to True to download to your computer

if DOWNLOAD_LOCALLY and final_output.exists():
    from google.colab import files
    files.download(str(final_output))
    print('📥 Download started!')
else:
    print(f'📁 Your video is saved on Google Drive:')
    print(f'   My Drive/rotbots_workshop/final_video.mp4')
    print(f'\n   Set DOWNLOAD_LOCALLY = True to download to your computer.')

---
## 🎤 Station 7: Screening & Critique

Share your video with the group!

**Discussion questions:**
- How does your video compare to others made with the same pipeline?
- Where do you see bias in the content generation?
- Who is the "author" of this video?
- What does this mean for the preservation of digital art?
- Could you tell this was AI-generated?

---

## 📊 Pipeline Summary

Here's everything the machine did to create your video:

In [ ]:
# PIPELINE SUMMARY
print('📊 ROTBOTS Pipeline Summary')
print('=' * 60)

# Load all data
steps = []
if (WORK_DIR / 'summaries.json').exists():
    with open(WORK_DIR / 'summaries.json') as f:
        s = json.load(f)
    steps.append(f'📥 Sources: {len(s.get("sources", []))} scraped & summarized')

if (WORK_DIR / 'essay_script.json').exists():
    with open(WORK_DIR / 'essay_script.json') as f:
        e = json.load(f)
    ch_count = len(e.get('chapters', []))
    sec_count = sum(len(c.get('sections',[])) for c in e.get('chapters',[]))
    steps.append(f'✍️ Essay: "{e.get("title", "Untitled")}" — {ch_count} chapters, {sec_count} sections')

if (WORK_DIR / 'storyboard.json').exists():
    with open(WORK_DIR / 'storyboard.json') as f:
        sb = json.load(f)
    steps.append(f'🎬 Storyboard: {len(sb)} scenes')

if (WORK_DIR / 'prompts.json').exists():
    with open(WORK_DIR / 'prompts.json') as f:
        p = json.load(f)
    steps.append(f'🎥 Video prompts: {len(p)} T2V prompts generated')

steps.append(f'🎙️ Narration: {len(audio_files)} audio files')
steps.append(f'🎬 Video clips: {len(video_files)} generated/uploaded')

if final_output.exists():
    dur = get_duration(final_output)
    size = final_output.stat().st_size / (1024*1024)
    steps.append(f'🎞️ Final video: {dur:.1f}s, {size:.1f}MB')

for step in steps:
    print(f'   {step}')

print(f'\n{"=" * 60}')
print(f'\n🤖 All of this was generated by AI from a single topic.')
print(f'   The only human input was: choosing the topic and pressing Play.')

---

## 🏁 That's it!

You've built and operated an AI content machine.

From a single topic, the pipeline:
1. Scraped and summarized sources
2. Wrote a structured essay with narration
3. Translated text into visual video prompts
4. Generated a synthetic voice
5. Created AI video clips
6. Assembled everything into a finished video

**Every step involved automated decisions — about what's important, what looks good, what sounds right, and how to tell the story.**

The question isn't whether AI can make content. It clearly can.

The question is: **what does that mean?**

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026, Amsterdam*